In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)

In [2]:
SEED = 42

In [3]:
cwd = Path.cwd()
table_dir = cwd.parent / 'data' / 'tables'

In [4]:
# COMBINE FLUX COLUMNS

# add all tables to one df
tables = table_dir.rglob('*')
df = pd.concat((pd.read_csv(t) for t in tables), ignore_index=True)

# extract CEERS and PRIMER fluxes in the same units
ceers_flux = df.filter(regex='^FLUX_')
ceers_err = df.filter(regex='^FLUXERR_')
primer_flux = df.filter(regex='^f_f') * 10
primer_err = df.filter(regex='^e_f') * 10

# drop the survey specific fluxes from the df
df.drop((list(ceers_flux.columns) + list(primer_flux.columns)
         + list(ceers_err.columns) + list(primer_err.columns)),
         inplace=True, axis=1)

# rename extracted flux columns to match eachother
ceers_flux.rename(columns=lambda x: x.replace('FLUX_', 'F'), inplace=True)
primer_flux.rename(columns=lambda x: x.replace('f_f', 'F')[:-1], inplace=True)
ceers_err.rename(columns=lambda x: x.replace('FLUXERR_', 'E'), inplace=True)
primer_err.rename(columns=lambda x: x.replace('e_f', 'E')[:-1], inplace=True)

# recombine the fluxes into one df
combined_flux = ceers_flux.combine_first(primer_flux)
combined_err = ceers_err.combine_first(primer_err)

# add fluxes back to main df
df = pd.concat([df, combined_flux, combined_err], axis=1)
print('Num galaxies (expect 486): ', len(df))

Num galaxies (expect 486):  486


In [5]:
# split into 2 dfs: one for candidates and one for non candidates
mask = (df['flag'] == 1.0)
dusty_df = df[mask]
nondusty_df = df[~mask]

dusty_df['flag'].head()

0     1
4     1
7     1
8     1
16    1
Name: flag, dtype: int64

In [6]:
# shuffle dfs
dusty_df = dusty_df.sample(frac=1, random_state=SEED).reset_index(drop=True)
nondusty_df = nondusty_df.sample(frac=1, random_state=SEED).reset_index(drop=True)

# split dfs
dusty_groups = dusty_df.index % 10
nondusty_groups = nondusty_df.index % 10
final_dusty = [group for _, group in dusty_df.groupby(dusty_groups)]
final_nondusty = [group for _, group in nondusty_df.groupby(nondusty_groups)]

# recombine dusty and nondusty dfs
final_dfs = []
for i, (dusty, nondusty) in enumerate(zip(final_dusty, final_nondusty)):
    print(f'idx={i}',
          f' dusty: {len(dusty)}',
          f' nondusty: {len(nondusty)}')
    final_dfs.append(pd.concat([dusty, nondusty]))

print('Num galaxies (expect 486): ', len(pd.concat(final_dfs)))

idx=0  dusty: 17  nondusty: 33
idx=1  dusty: 17  nondusty: 33
idx=2  dusty: 16  nondusty: 33
idx=3  dusty: 16  nondusty: 33
idx=4  dusty: 16  nondusty: 32
idx=5  dusty: 16  nondusty: 32
idx=6  dusty: 16  nondusty: 32
idx=7  dusty: 16  nondusty: 32
idx=8  dusty: 16  nondusty: 32
idx=9  dusty: 16  nondusty: 32
Num galaxies (expect 486):  486


In [7]:
# SAVE DFS AS CSVS

# create directory
dir = cwd.parent / 'train val test tables'
dir.mkdir(parents=True, exist_ok=True)

# save train and val sets
for i, df in enumerate(final_dfs[0:-1]):
    df.to_csv(dir / f'train_val_{i}.csv')

test_df = final_dfs[-1]
test_df.to_csv(dir / 'test.csv')